# 02 - Bronze Data Validation

## Objetivo
Validar la información almacenada en la capa Bronze,
comprobar la integridad de los datos y revisar la trazabilidad
de la ingesta realizada mediante Auto Loader.

## Actividades
1. Validación del volumen de registros.
2. Identificación de registros incompletos.
3. Revisión de duplicados.
4. Análisis del historial de Delta Lake.
5. Inspección del checkpoint de Auto Loader.

In [0]:
SELECT*
FROM ecomotive.bronze.iot_sensors_raw



sensor_id,truck_id,timestamp,temperature_c,latitude,longitude,status,battery_level_pct,firmware_version,signal_strength_dbm,_rescued_data
S-1001,T-101,2025-01-12T08:00:00,2.5,40.4168,-3.7038,OK,null,null,null,null
S-1001,T-101,2025-01-12T08:15:00,2.8,40.4200,-3.7100,OK,null,null,null,null
S-1001,T-101,2025-01-12T08:30:00,3.1,40.4300,-3.7200,OK,null,null,null,null
S-1002,T-102,2025-01-12T09:00:00,3.9,41.3851,2.1734,WARNING,null,null,null,null
S-1002,T-102,2025-01-12T09:15:00,4.2,41.3900,2.1800,ALARM,null,null,null,null
S-1002,T-102,2025-01-12T09:30:00,4.5,41.3950,2.1850,ALARM,null,null,null,null
S-1003,T-103,2025-01-12T08:10:00,-1.0,39.4699,-0.3763,OK,null,null,null,null
S-1004,T-104,2025-01-12T10:00:00,2.0,37.3891,-5.9845,OK,null,null,null,null
S-1005,T-105,2025-01-12T11:00:00,2.2,43.2630,-2.9350,OK,null,null,null,null
S-1001,T-101,2025-01-12T08:45:00,3.0,40.4400,-3.7300,OK,null,null,null,null


In [0]:
DESCRIBE TABLE  ecomotive.bronze.iot_sensors_raw

col_name,data_type,comment
sensor_id,string,null
truck_id,string,null
timestamp,string,null
temperature_c,string,null
latitude,string,null
longitude,string,null
status,string,null
battery_level_pct,string,null
firmware_version,string,null
signal_strength_dbm,string,null


In [0]:
SELECT 
    COUNT(*) as total_registros,
    COUNT(DISTINCT sensor_id) as sensores_unicos,
    COUNT(*) - COUNT(sensor_id) as registros_sin_sensor
FROM ecomotive.bronze.iot_sensors_raw

total_registros,sensores_unicos,registros_sin_sensor
103,37,7


In [0]:
---Registros Incompletos
SELECT
   COUNT(*) AS total_registros,
    SUM(CASE WHEN NULLIF(TRIM(sensor_id), '') IS NULL THEN 1 ELSE 0 END) AS sensor_id_sin_dato,
    SUM(CASE WHEN NULLIF(TRIM(truck_id), '') IS NULL THEN 1 ELSE 0 END) AS truck_id_sin_dato,
    SUM(CASE WHEN NULLIF(TRIM(timestamp), '') IS NULL THEN 1 ELSE 0 END) AS timestamp_sin_dato,
    SUM(CASE WHEN NULLIF(TRIM(temperature_c), '') IS NULL THEN 1 ELSE 0 END) AS temperatura_sin_dato,
    SUM(CASE WHEN NULLIF(TRIM(latitude), '') IS NULL THEN 1 ELSE 0 END) AS latitud_sin_dato,
    SUM(CASE WHEN NULLIF(TRIM(longitude), '') IS NULL THEN 1 ELSE 0 END) AS longitud_sin_dato,
    SUM(CASE WHEN NULLIF(TRIM(status), '') IS NULL THEN 1 ELSE 0 END) AS estado_sin_dato

FROM ecomotive.bronze.iot_sensors_raw;



total_registros,sensor_id_sin_dato,truck_id_sin_dato,timestamp_sin_dato,temperatura_sin_dato,latitud_sin_dato,longitud_sin_dato,estado_sin_dato
103,7,7,0,1,0,0,0


In [0]:
SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT sensor_id) AS sensores_unicos,
    COUNT(*) - COUNT(sensor_id) AS registros_sin_sensor
FROM ecomotive.bronze.iot_sensors_raw;

total_registros,sensores_unicos,registros_sin_sensor
103,37,7


In [0]:
-- Valiadción de duplicados

SELECT
    sensor_id,
    timestamp,
    COUNT(*)
FROM ecomotive.bronze.iot_sensors_raw
GROUP BY 1,2
HAVING count(*)>1

sensor_id,timestamp,COUNT(*)
S-1010,2025-01-12T13:00:00,3
S-1002,2025-01-12T09:30:00,3
S-1002,2025-01-12T09:45:00,3
S-1004,2025-01-12T10:00:00,3
S-1002,2025-01-12T09:00:00,4
S-1001,2025-01-12T08:45:00,3
S-1001,2025-01-12T08:15:00,4
S-1009,2025-01-12T12:45:00,3
S-1002,2025-01-12T09:15:00,4
S-1005,2025-01-12T11:00:00,3


In [0]:
-- Total Duplicados 

with duplicados as(
SELECT
    sensor_id,
    timestamp,
    row_number() over (Partition by sensor_id, timestamp ORDER BY timestamp ASC) as rank
FROM ecomotive.bronze.iot_sensors_raw
ORDER BY 1,2,3 asc)


select
count(*) as total_registro_adicionales
from duplicados
where rank >1




total_registro_adicionales
35


In [0]:
--Paso 4. Historial de Delta Lake

DESCRIBE HISTORY ecomotive.bronze.iot_sensors_raw;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
5,2026-11-09T02:12:01.000Z,75518753785960,dainercocap@outlook.com,STREAMING UPDATE,"Map(queryId -> 37779d09-32c7-4be2-bbb7-3e05b5f0c2a4, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 4, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(3992197074325286),d4eda68e-0550-4a24-9639-9b6e348b11b7,1108-193559-59mano4d-v2n,4,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 10, numOutputBytes -> 3663, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
4,2026-11-08T02:06:08.000Z,75518753785960,dainercocap@outlook.com,STREAMING UPDATE,"Map(queryId -> 37779d09-32c7-4be2-bbb7-3e05b5f0c2a4, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 3, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(3992197074325286),6037f17a-6e9b-4c6c-ad49-fb6006c31023,1107-231501-6vz51ffs-v2n,3,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 30, numOutputBytes -> 3773, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
3,2026-11-08T00:32:42.000Z,75518753785960,dainercocap@outlook.com,STREAMING UPDATE,"Map(queryId -> 37779d09-32c7-4be2-bbb7-3e05b5f0c2a4, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 2, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(3992197074325286),50b27d2f-1bb9-4589-8c54-7fae39b63f20,1107-231501-6vz51ffs-v2n,2,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 28, numOutputBytes -> 3692, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
2,2026-11-06T03:49:16.000Z,75518753785960,dainercocap@outlook.com,STREAMING UPDATE,"Map(queryId -> 37779d09-32c7-4be2-bbb7-3e05b5f0c2a4, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 1, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(3992197074325286),7e88839e-a03b-42f7-9001-df84f21d736e,1106-014851-9qvcy5e2-v2n,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 3, numOutputBytes -> 3269, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
1,2026-11-05T16:00:03.000Z,75518753785960,dainercocap@outlook.com,STREAMING UPDATE,"Map(queryId -> 37779d09-32c7-4be2-bbb7-3e05b5f0c2a4, outputMode -> Append, clusterBy -> [], statsOnLoad -> true, epochId -> 0, clusteringOnWriteStatus -> Reason for skipping: Not attempted due to eligibility heuristic)",null,List(3992197074325286),2e52baed-fdba-4b14-bc8c-2f35385fb4d1,1105-153207-lv77vcr6-v2n,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 32, numOutputBytes -> 3992, numAddedFiles -> 1)",null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13
0,2026-11-05T15:59:51.000Z,75518753785960,dainercocap@outlook.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""delta.enableRowTracking"":""true"",""delta.checkpointPolicy"":""v2"",""io.unitycatalog.tableId"":""6cceb44d-ee7d-417d-8879-f2d08b15a833"",""delta.rowTracking.materializedRowCommitVersionColumnName"":""_row-commit-version-col-e3db3e12-69f7-48a7-b772-d80b85e8dfb9"",""delta.rowTracking.materializedRowIdColumnName"":""_row-id-col-b02d5be7-d4f6-4d3c-9e83-7fe3e8cbca6c""}, statsOnLoad -> false, clusterByAuto -> true)",null,List(3992197074325286),2e52baed-fdba-4b14-bc8c-2f35385fb4d1,1105-153207-lv77vcr6-v2n,null,WriteSerializable,true,Map(),null,Databricks-Runtime/19.13.x-aarch64-photon-scala2.13


In [0]:
%python
checkpoint_path="/Volumes/ecomotive/metadata/checkpoints"
print(checkpoint_path)

files = dbutils.fs.ls(checkpoint_path)

print(f"Cantidad de elementos: {len(files)}")

display(files)

/Volumes/ecomotive/metadata/checkpoints
Cantidad de elementos: 5


path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/,commits/,0,1794190493462
dbfs:/Volumes/ecomotive/metadata/checkpoints/metadata,metadata,45,1793894394000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/,offsets/,0,1794190493462
dbfs:/Volumes/ecomotive/metadata/checkpoints/sensores_pipeline/,sensores_pipeline/,0,1794190493462
dbfs:/Volumes/ecomotive/metadata/checkpoints/sources/,sources/,0,1794190493462


In [0]:
%python

#Validación de los checkpoint
checkpoint_path = "/Volumes/ecomotive/metadata/checkpoints/"

display("offsets")
display(dbutils.fs.ls(checkpoint_path + "offsets/"))

display("Commits")
display(dbutils.fs.ls(checkpoint_path + "commits/"))


'offsets'

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/0,0,2735,1793894399000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/1,1,2736,1793936952000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/2,2,2736,1794097958000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/3,3,2736,1794103566000
dbfs:/Volumes/ecomotive/metadata/checkpoints/offsets/4,4,2736,1794190319000


'Commits'

path,name,size,modificationTime
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/0,0,29,1793894404000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/1,1,29,1793936957000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/2,2,29,1794097963000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/3,3,29,1794103569000
dbfs:/Volumes/ecomotive/metadata/checkpoints/commits/4,4,29,1794190322000


In [0]:
SELECT 

DISTINCT (
   truck_id
)
FROM ecomotive.bronze.iot_sensors_raw

truck_id
T-101
T-104
T-109
T-110
T-106
T-105
T-102
""
T-108
T-107


In [0]:
select count(*) from ecomotive.bronze.iot_sensors_raw

count(*)
103
